In [2]:
import numpy as np
from PIL import Image
import torch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
from torchvision import transforms
import os
from torch.utils.data import Dataset

In [ ]:
!wget https://github.com/SVizor42/ML_Zoomcamp/releases/download/straight-curly-data/data.zip
!unzip data.zip

In [6]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
class HairTypeClassifierCNN(nn.Module):
    def __init__(self):
        super(HairTypeClassifierCNN, self).__init__()

        self.feature_extractor = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            stride=1,
            padding=0
        )

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        self.flattened_features_size = 32 * 99 * 99

        self.classifier_head_fc1 = nn.Linear(self.flattened_features_size, 64)

        self.output_layer = nn.Linear(64, 1)

    def forward(self, image_data):

        features = self.feature_extractor(image_data)
        features = F.relu(features)
        features = self.pool(features)

        features_flat = features.view(features.size(0), -1)

        classification_result = self.classifier_head_fc1(features_flat)
        classification_result = F.relu(classification_result)

        output_logits = self.output_layer(classification_result)

        prediction_probability = torch.sigmoid(output_logits)

        return prediction_probability

hair_model = HairTypeClassifierCNN()

optimizer = optim.SGD(hair_model.parameters(), lr=0.002, momentum=0.8)
criterion = nn.BCEWithLogitsLoss()
print(hair_model)
print(f"\nOptimizer used: {type(optimizer).__name__} (LR={optimizer.param_groups[0]['lr']}, Momentum={optimizer.param_groups[0]['momentum']})")


HairTypeClassifierCNN(
  (feature_extractor): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (classifier_head_fc1): Linear(in_features=313632, out_features=64, bias=True)
  (output_layer): Linear(in_features=64, out_features=1, bias=True)
)

Optimizer used: SGD (LR=0.002, Momentum=0.8)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hair_model.to(device);

In [9]:
from torchsummary import summary
summary(hair_model, input_size=(3, 200, 200))

# # Option 2: Manual counting
# total_params = sum(p.numel() for p in model.parameters())
# print(f"Total parameters: {total_params}")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 198, 198]             896
         MaxPool2d-2           [-1, 32, 99, 99]               0
            Linear-3                   [-1, 64]      20,072,512
            Linear-4                    [-1, 1]              65
Total params: 20,073,473
Trainable params: 20,073,473
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.46
Forward/backward pass size (MB): 11.96
Params size (MB): 76.57
Estimated Total Size (MB): 89.00
----------------------------------------------------------------


In [10]:
class HairDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [11]:
input_size = 200

train_transforms = transforms.Compose([
    transforms.Resize((200, 200)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [12]:
from torch.utils.data import DataLoader

train_dataset = HairDataset(
    data_dir='./data/train',
    transform=train_transforms
)

validation_dataset = HairDataset(
    data_dir='./data/test',
    transform=val_transforms
)

train_loader = DataLoader(train_dataset, batch_size=20, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=20, shuffle=False)

In [13]:
num_epochs = 10
history = {'acc': [], 'loss': [], 'val_acc': [], 'val_loss': []}

for epoch in range(num_epochs):
    hair_model.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        labels = labels.float().unsqueeze(1)

        optimizer.zero_grad()
        outputs = hair_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predicted = (torch.sigmoid(outputs) > 0.5).float()
        total_train += labels.size(0)
        correct_train += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct_train / total_train
    history['loss'].append(epoch_loss)
    history['acc'].append(epoch_acc)

    hair_model.eval()
    val_running_loss = 0.0
    correct_val = 0
    total_val = 0
    with torch.no_grad():
        for images, labels in validation_loader:
            images, labels = images.to(device), labels.to(device)
            labels = labels.float().unsqueeze(1)

            outputs = hair_model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)
            predicted = (torch.sigmoid(outputs) > 0.5).float()
            total_val += labels.size(0)
            correct_val += (predicted == labels).sum().item()

    val_epoch_loss = val_running_loss / len(validation_dataset)
    val_epoch_acc = correct_val / total_val
    history['val_loss'].append(val_epoch_loss)
    history['val_acc'].append(val_epoch_acc)

    print(f"Epoch {epoch+1}/{num_epochs}, "
          f"Loss: {epoch_loss:.4f}, Acc: {epoch_acc:.4f}, "
          f"Val Loss: {val_epoch_loss:.4f}, Val Acc: {val_epoch_acc:.4f}")

Epoch 1/10, Loss: 0.6761, Acc: 0.4869, Val Loss: 0.6670, Val Acc: 0.4876
Epoch 2/10, Loss: 0.6545, Acc: 0.4869, Val Loss: 0.6526, Val Acc: 0.4876
Epoch 3/10, Loss: 0.6374, Acc: 0.4869, Val Loss: 0.6529, Val Acc: 0.4876
Epoch 4/10, Loss: 0.6324, Acc: 0.4869, Val Loss: 0.6939, Val Acc: 0.4876
Epoch 5/10, Loss: 0.6349, Acc: 0.4881, Val Loss: 0.6559, Val Acc: 0.4876
Epoch 6/10, Loss: 0.6203, Acc: 0.4869, Val Loss: 0.6538, Val Acc: 0.4925
Epoch 7/10, Loss: 0.6160, Acc: 0.4869, Val Loss: 0.6583, Val Acc: 0.4876
Epoch 8/10, Loss: 0.6071, Acc: 0.4869, Val Loss: 0.6654, Val Acc: 0.4876
Epoch 9/10, Loss: 0.6019, Acc: 0.4869, Val Loss: 0.6684, Val Acc: 0.4876
Epoch 10/10, Loss: 0.5966, Acc: 0.4869, Val Loss: 0.6899, Val Acc: 0.4876


In [14]:
history

{'acc': [0.4868913857677903,
  0.4868913857677903,
  0.4868913857677903,
  0.4868913857677903,
  0.48813982521847693,
  0.4868913857677903,
  0.4868913857677903,
  0.4868913857677903,
  0.4868913857677903,
  0.4868913857677903],
 'loss': [0.6761273581362545,
  0.6545137077681581,
  0.6374092925800366,
  0.6324426797445347,
  0.6349349484759175,
  0.6203225129999025,
  0.6159922395752611,
  0.6071467221750599,
  0.6019246454691322,
  0.5965635860978292],
 'val_acc': [0.48756218905472637,
  0.48756218905472637,
  0.48756218905472637,
  0.48756218905472637,
  0.48756218905472637,
  0.4925373134328358,
  0.48756218905472637,
  0.48756218905472637,
  0.48756218905472637,
  0.48756218905472637],
 'val_loss': [0.6670451385168293,
  0.6525555316785082,
  0.6529307983704468,
  0.6938788950146727,
  0.6558878937763954,
  0.6537611933786478,
  0.6583127575134163,
  0.6653764652672098,
  0.6683787900713546,
  0.6899389446671329]}

In [16]:
np.median(history['acc'])

np.float64(0.4868913857677903)

In [17]:
np.std(history['loss'])

np.float64(0.023411661406172315)

In [18]:
np.mean(history['val_loss'])

np.float64(0.6658066408254614)

In [22]:
history['val_acc'][5:]

[0.4925373134328358,
 0.48756218905472637,
 0.48756218905472637,
 0.48756218905472637,
 0.48756218905472637]

In [23]:
np.mean(history['val_acc'][5:])

np.float64(0.48855721393034823)